# Next-node prediction on small Newick subtrees

One node per sequence position; no node identities in the model. We serialize a
rooted, ordered tree in **preorder** and predict the next `(value, child_count)`
record with a small causal Transformer. Child counts preserve the complete
topology: a stack reconstructs each parent–child edge.

This is a **synthetic feature-learning experiment on real subtree shapes**.
The numbers are not biological measurements, and traversal order is not time.
Predicting a tree's serialization does not forecast a future biological addition.

The representation follows the established [preorder degree-sequence
encoding](https://drops.dagstuhl.de/storage/00lipics/lipics-vol154-stacs2020/LIPIcs.STACS.2020.22/LIPIcs.STACS.2020.22.pdf).
[Shiv and Quirk (2019)](https://papers.neurips.cc/paper_files/paper/2019/hash/6e0917469214d8fbd8c517dcdc6b8dcf-Abstract.html)
describe richer tree positional encodings; here we use learned depth and sibling
position embeddings, not a reproduction of their method.

**Run on your A100:** install the dependencies in the next cell if needed, place
`dataset/public-2023-12-25.all.nwk` beside this notebook's directory, select a GPU
kernel, and run all cells. Edit `Config.source_path` for another location.
The default is the full 1,024-shape experiment. For a quick execution check, set
`PROFILE = "smoke"` in the configuration cell. Smoke results are not a learning
claim. All functions are defined here; the old notebook is not imported.

In [ ]:
# Uncomment if the notebook environment needs dependencies. Keep your installed GPU PyTorch.
# %pip install "torch>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "biopython>=1.85,<2"
import copy
import hashlib
import heapq
import io
import json
import math
import os
import platform
import random
import re
import time
from contextlib import nullcontext
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import Bio
from Bio import Phylo
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from IPython.display import Markdown, display

PROFILE = os.environ.get("NEWICK_LM_PROFILE", "full")  # "full" or "smoke"

@dataclass(frozen=True)
class Config:
    seed: int = 42
    source_path: str = "dataset/public-2023-12-25.all.nwk"
    output_dir: str = "artifacts/newick_node_lm"
    min_nodes: int = 16
    max_nodes: int = 64
    n_shapes: int = 1024
    n_values: int = 8
    context_length: int = 65
    n_layers: int = 4
    d_model: int = 128
    n_heads: int = 4
    d_ff: int = 512
    dropout: float = 0.1
    batch_size: int = 64
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    max_epochs: int = 30
    patience: int = 5
    grad_clip: float = 1.0
    generation_trials: int = 100
    device: str = "auto"  # "cuda" or "cpu" to override

cfg = Config()
if PROFILE == "smoke":
    cfg = replace(cfg, n_shapes=64, max_epochs=2, patience=2,
                  batch_size=16, generation_trials=8)
elif PROFILE != "full":
    raise ValueError("PROFILE must be 'full' or 'smoke'.")
assert 2 <= cfg.min_nodes <= cfg.max_nodes and cfg.n_shapes >= 20
assert cfg.n_values >= 2 and cfg.context_length >= cfg.max_nodes + 1
assert cfg.d_model % cfg.n_heads == 0

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
torch.set_num_threads(min(4, torch.get_num_threads()))
device = torch.device(("cuda" if torch.cuda.is_available() else "cpu")
                      if cfg.device == "auto" else cfg.device)
USE_BF16 = device.type == "cuda" and torch.cuda.is_bf16_supported()

def autocast_context():
    return torch.autocast("cuda", dtype=torch.bfloat16) if USE_BF16 else nullcontext()

BASE_OUT = Path(cfg.output_dir)
OUT = BASE_OUT if PROFILE == "full" else BASE_OUT / "smoke"
OUT.mkdir(parents=True, exist_ok=True)

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)

environment = dict(python=platform.python_version(), torch=str(torch.__version__),
                   numpy=np.__version__, biopython=Bio.__version__,
                   matplotlib=matplotlib.__version__, device=str(device),
                   precision="bfloat16" if USE_BF16 else "float32", profile=PROFILE)
if device.type == "cuda":
    environment["gpu"] = torch.cuda.get_device_name(device)
save_json(OUT / "config.json", asdict(cfg))
save_json(OUT / "environment.json", environment)
print(json.dumps(environment, indent=2))
print(f"Outputs: {OUT.resolve()}")
print("Seeds are fixed; floating-point results may vary between devices.")


## 1. Extract small, complete clades

The full file has about 9.54 million nodes and includes multifurcations. We never
construct that entire tree in memory. A streaming structural scan selects
disjoint complete clades with 16–64 nodes, then Biopython parses only the sampled
fragments. The scanner rejects quoted labels and comments, which are absent
from this input. It is deliberately a restricted extractor, not a general Newick parser.

At each closing clade, select it if it meets the size bounds and contains no
previously selected clade. Sampling is uniform over the **distinct unordered
shapes produced by this selection policy**, not over all possible clades in the
original tree. Seeded hash priorities select the sample. Original child order
is preserved in each selected example.

Shape signatures sort child signatures only for deduplication. Names, branch
lengths, and permutations of siblings cannot cause the same shape to cross
splits. The cache is checked against a SHA-256 fingerprint of the source file.

In [ ]:
STRUCTURAL = re.compile(rb"[(),;]")
CACHE_VERSION = 1

def file_fingerprint(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return {"bytes": Path(path).stat().st_size, "sha256": digest.hexdigest()}

def scan_clades(path, min_nodes, max_nodes, sample_size, seed, block_size=8*1024*1024):
    # Frames contain only bounded shape descriptions, even for very large ancestors.
    stack, seen, heap = [], set(), []
    pending, trees, eligible, offset = False, 0, 0, 0
    started = False

    def add_child(frame, size, signature, covered):
        frame["size"] += size
        frame["covered"] |= covered
        if frame["size"] > max_nodes or frame["covered"]:
            frame["children"] = None
        elif frame["children"] is not None:
            frame["children"].append(signature)

    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            if any(mark in block for mark in (b"'", b"[", b"]")):
                raise ValueError("Extractor does not support quoted labels or comments.")
            if trees and block.strip():
                raise ValueError("Expected exactly one tree with no trailing content.")
            for match in STRUCTURAL.finditer(block):
                char, position = match[0], offset + match.start()
                if trees:
                    raise ValueError("Expected exactly one tree.")
                if char == b"(":
                    if started and not pending:
                        raise ValueError("Unexpected opening parenthesis.")
                    started = True
                    stack.append(dict(start=position, size=1, children=[], covered=False))
                    pending = True
                elif char == b",":
                    if not stack:
                        raise ValueError("Comma outside a clade.")
                    if pending:
                        add_child(stack[-1], 1, "()", False)
                    pending = True
                elif char == b")":
                    if not stack:
                        raise ValueError("Unmatched closing parenthesis.")
                    if pending:
                        add_child(stack[-1], 1, "()", False)
                    frame = stack.pop()
                    signature = ("(" + "".join(sorted(frame["children"])) + ")"
                                 if frame["children"] is not None else None)
                    if not frame["covered"] and min_nodes <= frame["size"] <= max_nodes:
                        eligible += 1
                        if signature not in seen:
                            seen.add(signature)
                            priority = int.from_bytes(hashlib.sha256(
                                f"{seed}:{signature}".encode()).digest(), "big")
                            entry = (-priority, signature, frame["start"], position + 1)
                            if len(heap) < sample_size:
                                heapq.heappush(heap, entry)
                            elif entry > heap[0]:
                                heapq.heapreplace(heap, entry)
                        frame["covered"] = True
                    if stack:
                        add_child(stack[-1], frame["size"], signature, frame["covered"])
                    pending = False
                else:
                    if stack or not started:
                        raise ValueError("Unclosed clade or missing tree at semicolon.")
                    trees += 1
                    if block[match.end():].strip():
                        raise ValueError("Unexpected content after the tree terminator.")
            offset += len(block)
    if stack or trees != 1:
        raise ValueError("Expected one complete, semicolon-terminated tree.")
    if len(heap) < sample_size:
        raise ValueError(f"Found {len(heap)} distinct shapes; requested {sample_size}.")
    selected = [dict(signature=sig, start=start, body_end=end)
                for _, sig, start, end in sorted(heap, reverse=True)]
    return selected, {"eligible_disjoint_clades": eligible, "distinct_shapes": len(seen)}

def read_fragment(handle, start, body_end):
    handle.seek(start)
    body = handle.read(body_end - start)
    suffix = bytearray()
    while True:
        part = handle.read(256)
        if not part:
            raise ValueError("Missing delimiter after a selected clade.")
        delimiter = STRUCTURAL.search(part)
        if delimiter:
            suffix.extend(part[:delimiter.start()])
            break
        suffix.extend(part)
    return (body + bytes(suffix) + b";").decode("utf-8")

def clade_signature(clade):
    return "(" + "".join(sorted(clade_signature(c) for c in clade.clades)) + ")"

def load_subtree_pool(path, sample_size, seed, min_nodes, max_nodes):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Place the Newick file at {path.resolve()}, or edit cfg.source_path.")
    fingerprint = file_fingerprint(path)
    specification = dict(version=CACHE_VERSION, fingerprint=fingerprint,
                         sample_size=sample_size, seed=seed,
                         min_nodes=min_nodes, max_nodes=max_nodes)
    cache_path = BASE_OUT / f"subtrees_{sample_size}_{seed}_{min_nodes}_{max_nodes}.json"
    if cache_path.exists():
        cached = json.loads(cache_path.read_text())
        if cached.get("specification") == specification:
            print(f"Loaded {len(cached['shapes'])} cached subtree shapes.")
            return cached
    print("Scanning the full file for distinct small clades…", flush=True)
    selected, census = scan_clades(path, min_nodes, max_nodes, sample_size, seed)
    shapes = []
    with path.open("rb") as handle:
        for entry in selected:
            fragment = read_fragment(handle, entry["start"], entry["body_end"])
            tree = Phylo.read(io.StringIO(fragment), "newick")
            nodes = list(tree.find_clades(order="preorder"))
            signature = clade_signature(tree.root)
            if signature != entry["signature"]:
                raise ValueError("Streaming scanner and Biopython disagree on a clade.")
            shapes.append(dict(
                shape_id=hashlib.sha256(signature.encode()).hexdigest(),
                signature=signature, counts=[len(n.clades) for n in nodes],
                source_start=entry["start"], source_body_end=entry["body_end"],
                source_newick=fragment,
                metadata=dict(names=[n.name for n in nodes],
                              branch_lengths=[n.branch_length for n in nodes])))
    result = dict(specification=specification, census=census, shapes=shapes)
    save_json(cache_path, result)
    print(f"Cached {len(shapes)} shapes. Census: {census}")
    return result

# The smoke profile shares the full extraction cache, then uses a small subset.
pool = load_subtree_pool(cfg.source_path, max(1024, cfg.n_shapes), cfg.seed,
                        cfg.min_nodes, cfg.max_nodes)
shapes = pool["shapes"][:cfg.n_shapes]
print(f"Experiment: {len(shapes)} shapes; sizes "
      f"{min(len(s['counts']) for s in shapes)}–{max(len(s['counts']) for s in shapes)} nodes.")


## 2. Node records, topology, and split isolation

Values are in `0..7`; child counts are in `0..63`. The joint vocabulary has
512 **reusable feature combinations**, with `token_id = 8 * child_count + value`.
BOS and PAD are input-only control symbols. There is no learned identity for a
sample, clade, or source node.

For each shape, set the root to each of the eight possible values. Every other
node has `value = (parent_value + 1-based sibling_position) % 8`. Split shapes
before constructing these examples. All eight versions of a shape stay together.

The stack below derives parent, depth, and sibling position only from the
records already emitted. A **subtree-return transition** means that one or more
internal subtrees have just finished and the next node begins another branch;
ordinary consecutive leaf siblings do not count.

In [ ]:
@dataclass(frozen=True)
class NodeRecord:
    value: int
    child_count: int

@dataclass
class FeatureNode:
    value: int
    children: list = field(default_factory=list)

VOCAB_SIZE = cfg.n_values * cfg.max_nodes
BOS_ID, PAD_ID = VOCAB_SIZE, VOCAB_SIZE + 1

def record_id(record):
    if not isinstance(record, NodeRecord):
        raise TypeError("Expected NodeRecord(value, child_count).")
    if type(record.value) is not int or type(record.child_count) is not int:
        raise ValueError("Node fields must be integers.")
    if not 0 <= record.value < cfg.n_values or not 0 <= record.child_count < cfg.max_nodes:
        raise ValueError("Node fields are outside the configured vocabulary.")
    return cfg.n_values * record.child_count + record.value

def id_record(token):
    token = int(token)
    if not 0 <= token < VOCAB_SIZE:
        raise ValueError("Only node token IDs can be decoded as records.")
    return NodeRecord(token % cfg.n_values, token // cfg.n_values)

def prefix_geometry(records, require_complete=False):
    if len(records) > cfg.max_nodes:
        raise ValueError("Too many nodes for this experiment.")
    parents, depths, siblings, returns, stack = [], [], [], [], []
    returned = False
    for i, record in enumerate(records):
        record_id(record)
        if i == 0:
            parent, depth, sibling = -1, 0, 0
        else:
            if not stack:
                raise ValueError("Extra node after a complete tree.")
            parent, remaining = stack[-1]
            depth = depths[parent] + 1
            sibling = records[parent].child_count - remaining + 1
            stack[-1][1] -= 1
        parents.append(parent); depths.append(depth); siblings.append(sibling)
        returns.append(returned)
        if record.child_count:
            stack.append([i, record.child_count])
        popped = 0
        while stack and stack[-1][1] == 0:
            stack.pop(); popped += 1
        returned = bool(popped and stack)
    slots = sum(remaining for _, remaining in stack) if records else 1
    if len(records) + slots > cfg.max_nodes:
        raise ValueError("Prefix cannot be completed within max_nodes.")
    complete = bool(records) and slots == 0
    if require_complete and not complete:
        raise ValueError("Incomplete tree: child slots remain unfilled.")
    next_parent = stack[-1][0] if stack else -1
    next_depth = depths[next_parent] + 1 if stack else 0
    next_sibling = (records[next_parent].child_count - stack[-1][1] + 1) if stack else 0
    return dict(parents=parents, depths=depths, siblings=siblings, returns=returns,
                slots=slots, complete=complete, next_parent=next_parent,
                next_depth=next_depth, next_sibling=next_sibling)

def encode_tree(root):
    records, stack = [], [root]
    while stack:
        node = stack.pop()
        records.append(NodeRecord(node.value, len(node.children)))
        stack.extend(reversed(node.children))
    prefix_geometry(records, require_complete=True)
    return records

def decode_records(records):
    geometry = prefix_geometry(records, require_complete=True)
    nodes = [FeatureNode(r.value) for r in records]
    for i, parent in enumerate(geometry["parents"]):
        if parent >= 0:
            nodes[parent].children.append(nodes[i])
    return nodes[0]

def assign_features(counts, root_value):
    template = [NodeRecord(0, int(k)) for k in counts]
    geometry = prefix_geometry(template, require_complete=True)
    values = [int(root_value)]
    for parent, sibling in zip(geometry["parents"][1:], geometry["siblings"][1:]):
        values.append((values[parent] + sibling) % cfg.n_values)
    return [NodeRecord(value, count) for value, count in zip(values, counts)]

rng = np.random.default_rng(cfg.seed)
order = rng.permutation(len(shapes)).tolist()
train_end, valid_end = int(0.8 * len(order)), int(0.9 * len(order))
split_indices = dict(train=order[:train_end], validation=order[train_end:valid_end],
                     test=order[valid_end:])
split_ids = {name: [shapes[i]["shape_id"] for i in indices]
             for name, indices in split_indices.items()}
examples = {
    name: [dict(shape_id=shapes[i]["shape_id"], root_value=value,
                records=assign_features(shapes[i]["counts"], value))
           for i in indices for value in range(cfg.n_values)]
    for name, indices in split_indices.items()
}
save_json(OUT / "split_ids.json", split_ids)
save_json(OUT / "examples.json", {
    name: [dict(shape_id=row["shape_id"], root_value=row["root_value"],
                token_ids=[record_id(r) for r in row["records"]]) for row in rows]
    for name, rows in examples.items()
})
save_json(OUT / "encoding.json", dict(version=1, n_values=cfg.n_values,
    max_nodes=cfg.max_nodes, vocab_size=VOCAB_SIZE, bos_id=BOS_ID, pad_id=PAD_ID,
    node_id_formula="n_values * child_count + value", traversal="preorder"))
for name in examples:
    print(f"{name:10s}: {len(split_ids[name]):4d} shapes, {len(examples[name]):5d} trees")

def collate(rows):
    longest = max(len(row["records"]) for row in rows)
    if longest > cfg.context_length:
        raise ValueError("Example exceeds context; no truncation is performed.")
    shape = (len(rows), longest)
    batch = dict(input_ids=torch.full(shape, PAD_ID, dtype=torch.long),
                 depths=torch.zeros(shape, dtype=torch.long),
                 siblings=torch.zeros(shape, dtype=torch.long),
                 targets=torch.full(shape, -100, dtype=torch.long),
                 return_mask=torch.zeros(shape, dtype=torch.bool))
    for i, row in enumerate(rows):
        records = row["records"]
        geometry = prefix_geometry(records, require_complete=True)
        tokens = [record_id(r) for r in records]
        n = len(tokens)
        batch["input_ids"][i, :n] = torch.tensor([BOS_ID] + tokens[:-1])
        batch["targets"][i, :n] = torch.tensor(tokens)
        batch["depths"][i, :n] = torch.tensor([0] + geometry["depths"][:-1])
        batch["siblings"][i, :n] = torch.tensor([0] + geometry["siblings"][:-1])
        batch["return_mask"][i, :n] = torch.tensor(geometry["returns"])
    return batch

def to_device(batch, destination=device):
    return {key: value.to(destination) for key, value in batch.items()}

def make_loader(rows, shuffle=False):
    return DataLoader(rows, batch_size=cfg.batch_size, shuffle=shuffle, collate_fn=collate,
                      num_workers=0, generator=torch.Generator().manual_seed(cfg.seed))

loaders = {name: make_loader(rows, name == "train") for name, rows in examples.items()}


## 3. Verify the data contract

These checks cover the streaming extractor at byte boundaries, multifurcations,
shape isolation, round trips, the numeric rule, and the shifted targets. A
complete tree has exactly `node_count - 1` edges. An unfinished prefix must
have enough room left to fill all declared child slots.

In [ ]:
checks = {}

def expect_value_error(function):
    try:
        function()
    except ValueError:
        return
    raise AssertionError("Expected a ValueError.")

def check_data():
    chain = FeatureNode(0)
    for value in range(1, 8):
        chain = FeatureNode(value, [chain])
    fixtures = [FeatureNode(3), chain,
        FeatureNode(2, [FeatureNode(3), FeatureNode(4, [FeatureNode(5), FeatureNode(6)])]),
        FeatureNode(6, [FeatureNode(i) for i in range(6)])]
    for tree in fixtures:
        records = encode_tree(tree)
        assert decode_records(records) == tree
        assert encode_tree(decode_records(records)) == records
        assert sum(r.child_count for r in records) == len(records) - 1
        for length in range(len(records) + 1):
            prefix = prefix_geometry(records[:length])
            full = prefix_geometry(records)
            for key in ("parents", "depths", "siblings", "returns"):
                assert prefix[key] == full[key][:length]
    return_tree = FeatureNode(0, [FeatureNode(1, [FeatureNode(2), FeatureNode(3)]), FeatureNode(4)])
    assert prefix_geometry(encode_tree(return_tree))["returns"] == [False]*4 + [True]
    assert not any(prefix_geometry(encode_tree(fixtures[-1]))["returns"])
    expect_value_error(lambda: decode_records([NodeRecord(0, 2), NodeRecord(1, 0)]))
    expect_value_error(lambda: decode_records([NodeRecord(0, 0), NodeRecord(1, 0)]))
    expect_value_error(lambda: record_id(NodeRecord(8, 0)))
    expect_value_error(lambda: prefix_geometry([NodeRecord(0, 63), NodeRecord(0, 63)]))

    fixture_dir = OUT / "check_fixtures"
    fixture_dir.mkdir(exist_ok=True)
    path = fixture_dir / "extractor.nwk"
    path.write_text("((a:1,b:2)x:3,(c:1,d:2,e:3)y:4)r:0;\n")
    reference = scan_clades(path, 3, 7, 2, 42, block_size=4096)
    for block_size in (1, 7, 13):
        assert scan_clades(path, 3, 7, 2, 42, block_size=block_size) == reference
    with path.open("rb") as handle:
        for entry in reference[0]:
            parsed = Phylo.read(io.StringIO(read_fragment(handle, entry["start"], entry["body_end"])), "newick")
            assert clade_signature(parsed.root) == entry["signature"]
            assert parsed.root.name in {"x", "y"}
            assert parsed.root.branch_length in {3.0, 4.0}
    for malformed in ("('a,b',c)r;", "(a[note],b)r;", "(a,b)r", "(a,b;", "(a,b)r;extra"):
        path.write_text(malformed)
        expect_value_error(lambda: scan_clades(path, 2, 7, 1, 42, block_size=7))
    sig1 = clade_signature(Phylo.read(io.StringIO("(a:1,(b,c)x)r;"), "newick").root)
    sig2 = clade_signature(Phylo.read(io.StringIO("((e,d)y:99,f)z;"), "newick").root)
    assert sig1 == sig2

    groups = [set(ids) for ids in split_ids.values()]
    assert all(not groups[i] & groups[j] for i in range(3) for j in range(i+1, 3))
    assert len(set.union(*groups)) == len(shapes)
    assert len({s["signature"] for s in shapes}) == len(shapes)
    intervals = sorted((s["source_start"], s["source_body_end"]) for s in shapes)
    assert all(end <= start for (_, end), (start, _) in zip(intervals, intervals[1:]))
    for name, rows in examples.items():
        assert len(rows) == len(split_ids[name]) * cfg.n_values
        allowed = set(split_ids[name])
        for row in rows:
            records = row["records"]
            geometry = prefix_geometry(records, require_complete=True)
            assert row["shape_id"] in allowed
            assert cfg.min_nodes <= len(records) <= cfg.max_nodes
            assert encode_tree(decode_records(records)) == records
            assert records[0].value == row["root_value"]
            for i in range(1, len(records)):
                assert records[i].value == (records[geometry["parents"][i]].value
                                           + geometry["siblings"][i]) % cfg.n_values
    rows = [min(examples["train"], key=lambda r: len(r["records"])),
            max(examples["train"], key=lambda r: len(r["records"]))]
    batch = collate(rows)
    assert set(batch) == {"input_ids", "depths", "siblings", "targets", "return_mask"}
    for i, row in enumerate(rows):
        tokens = [record_id(r) for r in row["records"]]
        n = len(tokens)
        assert batch["input_ids"][i, :n].tolist() == [BOS_ID] + tokens[:-1]
        assert batch["targets"][i, :n].tolist() == tokens
        assert (batch["input_ids"][i, n:] == PAD_ID).all()
        assert (batch["targets"][i, n:] == -100).all()
    checks["data_contract"] = True
    print("PASS: extraction boundaries, round trips, shape splits, features, shifts, and padding.")

check_data()


## 4. Small causal Transformer

The feature encoder embeds value and child count independently. Learned depth,
sibling position, and sequence position embeddings are added to that vector.
Each node still occupies just one position. BOS has its own embedding; source
metadata is never passed to the network.

Four pre-normalized decoder blocks use causal scaled dot-product attention.
The output head predicts all 512 node types jointly, so it does not assume
independence between the next value and next child count. Right padding is
after all valid inputs and therefore invisible to earlier causal positions;
padded targets are excluded from loss.

In [ ]:
class NodeFeatureEncoder(nn.Module):
    """Replace or extend this module when adding real categorical/numeric features."""
    def __init__(self, config):
        super().__init__()
        self.value = nn.Embedding(config.n_values, config.d_model)
        self.child_count = nn.Embedding(config.max_nodes, config.d_model)

    def forward(self, values, child_counts):
        return self.value(values) + self.child_count(child_counts)

class DecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = config.n_heads
        self.width = config.d_model // config.n_heads
        self.dropout_p = config.dropout
        self.norm1 = nn.LayerNorm(config.d_model)
        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.projection = nn.Linear(config.d_model, config.d_model)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.mlp = nn.Sequential(nn.Linear(config.d_model, config.d_ff), nn.GELU(),
                                 nn.Linear(config.d_ff, config.d_model), nn.Dropout(config.dropout))
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        batch, length, width = x.shape
        qkv = self.qkv(self.norm1(x)).view(batch, length, 3, self.heads, self.width)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        attention = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout_p if self.training else 0.0)
        attention = attention.transpose(1, 2).contiguous().view(batch, length, width)
        x = x + self.dropout(self.projection(attention))
        return x + self.mlp(self.norm2(x))

class NodeTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.vocab_size = config.n_values * config.max_nodes
        self.features = NodeFeatureEncoder(config)
        self.special = nn.Embedding(2, config.d_model)  # BOS, PAD
        self.depth = nn.Embedding(config.max_nodes, config.d_model)
        self.sibling = nn.Embedding(config.max_nodes, config.d_model)
        self.position = nn.Embedding(config.context_length, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layers)])
        self.norm = nn.LayerNorm(config.d_model)
        self.head = nn.Linear(config.d_model, self.vocab_size, bias=False)
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, batch):
        ids = batch["input_ids"]
        if ids.size(1) > self.config.context_length:
            raise ValueError("Input exceeds configured context length.")
        values = ids.remainder(self.config.n_values)
        counts = (ids // self.config.n_values).clamp(max=self.config.max_nodes - 1)
        x = self.features(values, counts)
        controls = self.special((ids - self.vocab_size).clamp(0, 1))
        x = torch.where((ids < self.vocab_size).unsqueeze(-1), x, controls)
        positions = torch.arange(ids.size(1), device=ids.device)
        x = self.dropout(x + self.depth(batch["depths"]) + self.sibling(batch["siblings"])
                         + self.position(positions)[None])
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))

def next_node_loss(logits, targets):
    return F.cross_entropy(logits.float().reshape(-1, VOCAB_SIZE), targets.reshape(-1),
                           ignore_index=-100)

seed_everything(cfg.seed)
model = NodeTransformer(cfg).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## 5. Causality, padding, and optimizer checks

The overfitting check deliberately memorizes **one training tree** using a
separate model. It only verifies the optimization machinery. All evidence of
generalization comes from the shape-disjoint validation/test experiment below.

In [ ]:
@torch.no_grad()
def check_model_math(net):
    net.eval()
    rows = [min(examples["train"], key=lambda r: len(r["records"])),
            max(examples["train"], key=lambda r: len(r["records"]))]
    batch = to_device(collate(rows))
    logits = net(batch)
    changed = {key: value.clone() for key, value in batch.items()}
    cut = 4
    suffix = changed["input_ids"][:, cut:]
    changed["input_ids"][:, cut:] = torch.where(
        suffix < VOCAB_SIZE,
        (suffix // cfg.n_values) * cfg.n_values + (suffix + 1) % cfg.n_values, suffix)
    torch.testing.assert_close(net(changed)[:, :cut], logits[:, :cut], rtol=1e-5, atol=1e-6)
    for i, row in enumerate(rows):
        single = net(to_device(collate([row])))
        torch.testing.assert_close(single[0], logits[i, :len(row["records"])], rtol=1e-4, atol=1e-5)
    valid = batch["targets"] != -100
    direct = F.cross_entropy(logits[valid].float(), batch["targets"][valid])
    torch.testing.assert_close(next_node_loss(logits, batch["targets"]), direct)
    checks["causality_and_padding"] = True
    print("PASS: future-token perturbations, batch padding, and masked loss.")

def check_tiny_overfit():
    seed_everything(cfg.seed + 1)
    net = NodeTransformer(replace(cfg, dropout=0.0)).to(device)
    row = min(examples["train"], key=lambda r: len(r["records"]))
    batch = to_device(collate([row]))
    optimizer = torch.optim.AdamW(net.parameters(), lr=3e-3, weight_decay=0.0)
    final_loss, accuracy = math.inf, 0.0
    for step in range(1, 201):
        net.train()
        optimizer.zero_grad(set_to_none=True)
        loss = next_node_loss(net(batch), batch["targets"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.grad_clip)
        optimizer.step()
        if step % 10 == 0:
            net.eval()
            with torch.no_grad():
                logits = net(batch)
                final_loss = next_node_loss(logits, batch["targets"]).item()
                accuracy = (logits.argmax(-1) == batch["targets"]).float().mean().item()
            if final_loss < 0.05 and accuracy == 1.0:
                break
    if final_loss >= 0.05 or accuracy != 1.0:
        raise AssertionError(f"Tiny overfit failed: loss={final_loss:.4f}, accuracy={accuracy:.3f}")
    checks["tiny_overfit"] = dict(passed=True, steps=step, loss=final_loss, accuracy=accuracy)
    print(f"PASS: one-tree overfit in {step} steps; loss={final_loss:.4f}, accuracy={accuracy:.1%}.")
    del net, optimizer
    seed_everything(cfg.seed)

check_model_math(model)
check_tiny_overfit()
save_json(OUT / "checks.json", checks)


## 6. Train and select using validation only

Loss scores the unmasked distribution over all 512 node types. We do not apply
tree-validity masks or the arithmetic rule during training or likelihood
evaluation. The selected checkpoint minimizes validation joint next-node loss.

Accuracies for value and child count use their **marginal distributions**;
joint accuracy uses the highest-probability pair. Losses and accuracies are
weighted by the number of relevant nodes. The root's value is unpredictable
from BOS, so non-root results are reported separately.

In [ ]:
class MetricTotals:
    def __init__(self):
        self.sums = dict(nodes=0, trees=0, joint_nll=0.0, joint_correct=0,
                         feature_correct=0, child_count_correct=0,
                         non_root_nodes=0, non_root_feature_correct=0, non_root_feature_nll=0.0,
                         return_nodes=0, return_feature_correct=0, return_feature_nll=0.0)

    @torch.no_grad()
    def update(self, logp, batch):
        target = batch["targets"]
        valid = target != -100
        safe = target.clamp(min=0)
        value, count = safe % cfg.n_values, safe // cfg.n_values
        joint_nll = -logp.gather(-1, safe.unsqueeze(-1)).squeeze(-1)
        joint = logp.view(*logp.shape[:-1], cfg.max_nodes, cfg.n_values)
        value_logp = torch.logsumexp(joint, dim=-2)
        count_logp = torch.logsumexp(joint, dim=-1)
        value_ok = value_logp.argmax(-1) == value
        value_nll = -value_logp.gather(-1, value.unsqueeze(-1)).squeeze(-1)
        self.sums["nodes"] += int(valid.sum())
        self.sums["trees"] += target.size(0)
        self.sums["joint_nll"] += float(joint_nll[valid].sum())
        self.sums["joint_correct"] += int(((logp.argmax(-1) == safe) & valid).sum())
        self.sums["feature_correct"] += int((value_ok & valid).sum())
        self.sums["child_count_correct"] += int(((count_logp.argmax(-1) == count) & valid).sum())
        non_root = valid & (torch.arange(target.size(1), device=target.device)[None] > 0)
        for name, mask in (("non_root", non_root), ("return", valid & batch["return_mask"])):
            self.sums[f"{name}_nodes"] += int(mask.sum())
            self.sums[f"{name}_feature_correct"] += int((value_ok & mask).sum())
            self.sums[f"{name}_feature_nll"] += float(value_nll[mask].sum())

    def result(self):
        s = self.sums
        result = dict(nodes=s["nodes"], trees=s["trees"], joint_nll=s["joint_nll"] / s["nodes"],
                      joint_accuracy=s["joint_correct"] / s["nodes"],
                      feature_accuracy=s["feature_correct"] / s["nodes"],
                      child_count_accuracy=s["child_count_correct"] / s["nodes"])
        for name in ("non_root", "return"):
            count = s[f"{name}_nodes"]
            result[f"{name}_nodes"] = count
            result[f"{name}_feature_accuracy"] = s[f"{name}_feature_correct"] / count if count else None
            result[f"{name}_feature_nll"] = s[f"{name}_feature_nll"] / count if count else None
        return result

@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    totals = MetricTotals()
    for batch in loader:
        batch = to_device(batch)
        with autocast_context():
            logits = net(batch)
        totals.update(F.log_softmax(logits.float(), dim=-1), batch)
    return totals.result()

def fit(net):
    seed_everything(cfg.seed)
    optimizer = torch.optim.AdamW(net.parameters(), lr=cfg.learning_rate,
                                  weight_decay=cfg.weight_decay)
    history, best_loss, stale = [], math.inf, 0
    started = time.monotonic()
    for epoch in range(1, cfg.max_epochs + 1):
        net.train()
        epoch_loss, epoch_nodes = 0.0, 0
        epoch_started = time.monotonic()
        for batch in loaders["train"]:
            batch = to_device(batch)
            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                logits = net(batch)
                loss = next_node_loss(logits, batch["targets"])
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite training loss.")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.grad_clip)
            optimizer.step()
            n = int((batch["targets"] != -100).sum())
            epoch_loss += loss.item() * n
            epoch_nodes += n
        validation = evaluate(net, loaders["validation"])
        row = dict(epoch=epoch, train_joint_nll=epoch_loss / epoch_nodes,
                   validation=validation, seconds=time.monotonic() - epoch_started)
        history.append(row)
        if validation["joint_nll"] < best_loss:
            best_loss, stale = validation["joint_nll"], 0
            best_state = {key: value.detach().cpu().clone() for key, value in net.state_dict().items()}
            torch.save(dict(version=1, config=asdict(cfg), state_dict=best_state,
                            best_epoch=epoch, validation=validation,
                            source_fingerprint=pool["specification"]["fingerprint"],
                            split_ids=split_ids, environment=environment), OUT / "checkpoint.pt")
        else:
            stale += 1
        save_json(OUT / "history.json", history)
        ret = validation["return_feature_accuracy"]
        ret_text = "n/a" if ret is None else f"{ret:.1%}"
        print(f"Epoch {epoch:02d} | train NLL {row['train_joint_nll']:.4f} | "
              f"val NLL {validation['joint_nll']:.4f} | "
              f"val return-value accuracy {ret_text} | {row['seconds']:.1f}s", flush=True)
        if stale >= cfg.patience:
            print("Early stopping.")
            break
    checkpoint = torch.load(OUT / "checkpoint.pt", map_location="cpu", weights_only=True)
    net.load_state_dict(checkpoint["state_dict"])
    net.eval()
    elapsed = time.monotonic() - started
    print(f"Restored epoch {checkpoint['best_epoch']}; total training time {elapsed:.1f}s.")
    return history, checkpoint["best_epoch"], elapsed

history, best_epoch, training_seconds = fit(model)


## 7. Compare with baselines on held-out shapes

The unigram baseline estimates training node frequencies. The bigram baseline
conditions on the immediately preceding node type (including BOS). Both use
add-one smoothing and assign probability to every one of the 512 node types.
Neither baseline sees validation or test counts.

The learning criterion requires **lower test joint NLL than both baselines**
and **higher feature accuracy on subtree-return transitions than both**.
Failure is reported directly. We do not tune on the test set or relax its split.
These are single-seed point estimates; eight value assignments of a shape are
correlated, so the number of independent shape groups is also shown.

In [ ]:
def fit_baselines(rows):
    unigram = torch.ones(VOCAB_SIZE, dtype=torch.float64)
    bigram = torch.ones(VOCAB_SIZE + 1, VOCAB_SIZE, dtype=torch.float64)
    for row in rows:
        tokens = [record_id(r) for r in row["records"]]
        previous = [BOS_ID] + tokens[:-1]
        for prev, token in zip(previous, tokens):
            unigram[token] += 1
            bigram[prev, token] += 1
    return dict(unigram=(unigram / unigram.sum()).log().float(),
                bigram=(bigram / bigram.sum(-1, keepdim=True)).log().float())

@torch.no_grad()
def evaluate_baseline(name, log_tables, loader):
    totals = MetricTotals()
    for batch in loader:
        if name == "unigram":
            logp = log_tables[name].expand(*batch["targets"].shape, VOCAB_SIZE)
        else:
            # Padded positions are ignored by the metric accumulator.
            logp = log_tables[name][batch["input_ids"].clamp(max=BOS_ID)]
        totals.update(logp, batch)
    return totals.result()

baseline_tables = fit_baselines(examples["train"])
test_metrics = {"transformer": evaluate(model, loaders["test"])}
test_metrics.update({name: evaluate_baseline(name, baseline_tables, loaders["test"])
                     for name in ("unigram", "bigram")})
transformer_metrics = test_metrics["transformer"]
joint_pass = all(transformer_metrics["joint_nll"] < test_metrics[name]["joint_nll"]
                 for name in ("unigram", "bigram"))
return_pass = (transformer_metrics["return_nodes"] > 0 and all(
    transformer_metrics["return_feature_accuracy"] > test_metrics[name]["return_feature_accuracy"]
    for name in ("unigram", "bigram")))
learning_check = dict(joint_nll_beats_both=joint_pass, return_feature_accuracy_beats_both=return_pass,
                      criterion_met=joint_pass and return_pass,
                      eligible_for_full_experiment_claim=PROFILE == "full")
results = dict(profile=PROFILE, test_shapes=len(split_ids["test"]), best_epoch=best_epoch,
               training_seconds=training_seconds, metrics=test_metrics,
               learning_check=learning_check)
save_json(OUT / "metrics.json", results)

def percent(value):
    return "n/a" if value is None else f"{100 * value:.1f}%"

table = ["| Method | Joint NLL ↓ | Joint accuracy | Value accuracy (non-root) | Child-count accuracy | Value accuracy (subtree returns) |",
         "|---|---:|---:|---:|---:|---:|"]
for name, metrics in test_metrics.items():
    table.append(f"| {name} | {metrics['joint_nll']:.4f} | {percent(metrics['joint_accuracy'])} | "
                 f"{percent(metrics['non_root_feature_accuracy'])} | {percent(metrics['child_count_accuracy'])} | "
                 f"{percent(metrics['return_feature_accuracy'])} |")
display(Markdown("\n".join(table)))
print(f"Test set: {len(split_ids['test'])} shape groups; {transformer_metrics['nodes']:,} node targets; "
      f"{transformer_metrics['return_nodes']:,} subtree-return targets.")
print("Learning criterion:", "PASS" if learning_check["criterion_met"] else "NOT MET")
if PROFILE == "smoke":
    print("SMOKE PROFILE: this only checks execution; it does not establish the full learning result.")

epochs = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [h["train_joint_nll"] for h in history], label="Train")
axes[0].plot(epochs, [h["validation"]["joint_nll"] for h in history], label="Validation")
axes[0].set(xlabel="Epoch", ylabel="Joint next-node NLL", title="Checkpoint selection")
for key, label in (("non_root_feature_accuracy", "All non-root nodes"),
                   ("return_feature_accuracy", "Subtree returns")):
    axes[1].plot(epochs, [h["validation"][key] for h in history], label=label)
axes[1].set(xlabel="Epoch", ylabel="Validation value accuracy", ylim=(0, 1), title="Feature learning")
for axis in axes:
    axis.axvline(best_epoch, color="gray", linestyle="--", label="Selected epoch")
    axis.legend(); axis.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUT / "training_curves.png", dpi=150)
plt.show()


## 8. Predict and generate node records

`predict_next(prefix)` returns the top joint node candidates and the next node's
attachment location, which the prefix already determines. The model still
predicts the next child count and value. Its probabilities remain unmasked.

`generate(prefix, constrained=True)` masks only child counts that would exceed
the total node budget. If `S` child slots remain and `n` nodes have been emitted,
choosing child count `k` leaves `S - 1 + k` slots after node `n + 1`. The mask
requires those slots to fit inside `max_nodes`. Every numeric value remains
available. No arithmetic rule is hard-coded into generation.

Generation stops when no child slots remain. Unconstrained generation uses the
same model without the budget mask; impossible-to-complete prefixes are reported
as invalid. Structural validity under the constrained decoder is guaranteed by
the decoder, **not evidence that the Transformer learned structure**. A sampled
continuation can differ from the held-out tree while remaining valid.

In [ ]:
def prefix_batch(records, destination):
    geometry = prefix_geometry(records)
    tokens = [BOS_ID] + [record_id(r) for r in records]
    return dict(input_ids=torch.tensor([tokens], dtype=torch.long, device=destination),
                depths=torch.tensor([[0] + geometry["depths"]], dtype=torch.long, device=destination),
                siblings=torch.tensor([[0] + geometry["siblings"]], dtype=torch.long, device=destination))

@torch.no_grad()
def next_logits(prefix, net=None):
    net = model if net is None else net
    if (net.config.n_values, net.config.max_nodes) != (cfg.n_values, cfg.max_nodes):
        raise ValueError("Checkpoint encoding differs from the active notebook configuration.")
    geometry = prefix_geometry(prefix)
    if geometry["complete"]:
        raise ValueError("This tree is already complete; no next node exists.")
    net.eval()
    destination = next(net.parameters()).device
    return net(prefix_batch(prefix, destination))[0, -1].float().cpu()

@torch.no_grad()
def predict_next(prefix, top_k=5, net=None):
    if not 1 <= top_k <= VOCAB_SIZE:
        raise ValueError("top_k must be within the node vocabulary.")
    geometry = prefix_geometry(prefix)
    probabilities = next_logits(prefix, net).softmax(-1)
    values, tokens = probabilities.topk(top_k)
    candidates = []
    for probability, token in zip(values.tolist(), tokens.tolist()):
        record = id_record(token)
        candidates.append(dict(value=record.value, child_count=record.child_count,
                               probability=probability,
                               fits_node_budget=len(prefix) + geometry["slots"] + record.child_count <= cfg.max_nodes))
    return dict(parent_preorder_index=geometry["next_parent"], depth=geometry["next_depth"],
                sibling_position=geometry["next_sibling"], candidates=candidates)

@torch.no_grad()
def generate(prefix=(), max_nodes=None, constrained=True, temperature=1.0, seed=42, net=None):
    limit = cfg.max_nodes if max_nodes is None else max_nodes
    if type(limit) is not int or not 1 <= limit <= cfg.max_nodes:
        raise ValueError("max_nodes must be an integer within the configured capacity.")
    if not math.isfinite(temperature) or temperature <= 0:
        raise ValueError("temperature must be finite and positive.")
    records = list(prefix)
    geometry = prefix_geometry(records)
    if len(records) + geometry["slots"] > limit:
        raise ValueError("The supplied prefix cannot be completed within max_nodes.")
    generator = torch.Generator(device="cpu").manual_seed(seed)
    counts = torch.arange(VOCAB_SIZE) // cfg.n_values
    while not geometry["complete"]:
        logits = next_logits(records, net) / temperature
        if constrained:
            feasible = counts <= limit - len(records) - geometry["slots"]
            logits = logits.masked_fill(~feasible, -torch.inf)
        token = int(torch.multinomial(logits.softmax(-1), 1, generator=generator))
        records.append(id_record(token))
        try:
            geometry = prefix_geometry(records)
        except ValueError as error:
            return dict(records=records, valid=False, reason=str(error), constrained=constrained)
        if len(records) + geometry["slots"] > limit:
            return dict(records=records, valid=False, reason="Declared children exceed the node budget.",
                        constrained=constrained)
    return dict(records=records, valid=True, reason="All child slots filled.", constrained=constrained)

def serializable_generation(result):
    return {**result, "records": [asdict(r) for r in result["records"]]}

def generated_feature_score(records, prefix_length):
    geometry = prefix_geometry(records, require_complete=True)
    indices = range(max(1, prefix_length), len(records))
    correct = sum(records[i].value == (records[geometry["parents"][i]].value
                  + geometry["siblings"][i]) % cfg.n_values for i in indices)
    count = max(0, len(records) - max(1, prefix_length))
    return correct, count

def evaluate_generation():
    rng = np.random.default_rng(cfg.seed + 100)
    selected = rng.choice(len(examples["test"]), min(cfg.generation_trials, len(examples["test"])),
                          replace=False).tolist()
    metrics, saved = {}, {}
    for constrained in (False, True):
        name = "constrained" if constrained else "unconstrained"
        valid, correct, count = 0, 0, 0
        samples = []
        for trial, index in enumerate(selected):
            row = examples["test"][index]
            prefix = row["records"][:min(8, len(row["records"]) - 1)]
            result = generate(prefix, constrained=constrained, seed=cfg.seed + trial)
            assert result["records"][:len(prefix)] == prefix
            if result["valid"]:
                valid += 1
                assert encode_tree(decode_records(result["records"])) == result["records"]
                c, n = generated_feature_score(result["records"], len(prefix))
                correct += c; count += n
            samples.append(dict(shape_id=row["shape_id"], root_value=row["root_value"],
                                prefix_length=len(prefix), **serializable_generation(result)))
        metrics[name] = dict(trials=len(selected), valid_completions=valid,
                             structural_validity=valid / len(selected),
                             generated_feature_accuracy_on_valid_completions=correct / count if count else None,
                             scored_generated_nodes=count)
        saved[name] = samples
        print(f"{name}: {valid}/{len(selected)} valid completions; "
              f"generated value accuracy on valid completions: {percent(correct / count if count else None)}")
    assert metrics["constrained"]["valid_completions"] == len(selected)
    checks["constrained_generation_validity"] = True
    save_json(OUT / "generation_samples.json", saved)
    return metrics, saved

generation_metrics, generation_samples = evaluate_generation()
results["generation"] = generation_metrics
save_json(OUT / "metrics.json", results)


## 9. Inspect an actual held-out prediction

The plot labels nodes with their synthetic value and child count. Blue nodes
are already visible, orange is the next target, and gray nodes have not yet
been supplied. Choose a subtree-return position when available. The displayed
future is for inspection only; `predict_next` receives just the blue prefix.

In [ ]:
def draw_records(records, prefix_length=0, title="Encoded tree"):
    geometry = prefix_geometry(records, require_complete=True)
    children = [[] for _ in records]
    for i, parent in enumerate(geometry["parents"]):
        if parent >= 0:
            children[parent].append(i)
    leaves = [i for i, kids in enumerate(children) if not kids]
    x = {node: float(rank) for rank, node in enumerate(leaves)}
    for i in reversed(range(len(records))):
        if children[i]:
            x[i] = float(np.mean([x[child] for child in children[i]]))
    fig, axis = plt.subplots(figsize=(min(18, max(9, len(leaves) * 0.8)),
                                      max(3.5, 1.1 * (max(geometry["depths"]) + 1))))
    for i, parent in enumerate(geometry["parents"]):
        if parent >= 0:
            axis.plot([x[parent], x[i]], [-geometry["depths"][parent], -geometry["depths"][i]],
                      color="#b8c4ce", zorder=1)
    for i, record in enumerate(records):
        color = "#d8eaff" if i < prefix_length else ("#ffd49b" if i == prefix_length else "#edf0f2")
        axis.text(x[i], -geometry["depths"][i], f"v={record.value}\nk={record.child_count}",
                  ha="center", va="center", fontsize=8,
                  bbox=dict(boxstyle="round,pad=0.3", facecolor=color, edgecolor="#687886"), zorder=2)
    axis.set_title(title, pad=20)
    axis.set_xlim(-0.8, max(x.values()) + 0.8)
    axis.set_ylim(-max(geometry["depths"]) - 0.7, 0.7)
    axis.axis("off"); fig.tight_layout()
    return fig

demo_rows = [row for row in examples["test"] if row["root_value"] == 3
             and any(prefix_geometry(row["records"])["returns"])]
demo = min(demo_rows or examples["test"], key=lambda row: len(row["records"]))
geometry = prefix_geometry(demo["records"])
prefix_length = (geometry["returns"].index(True) if any(geometry["returns"])
                 else min(8, len(demo["records"]) - 1))
prefix = demo["records"][:prefix_length]
prediction = predict_next(prefix)
actual = demo["records"][prefix_length]
print("Input prefix:", [(r.value, r.child_count) for r in prefix])
print("Next attachment (preorder index, not a learned identity):", prediction["parent_preorder_index"])
print("Observed next record:", actual)
candidate_table = ["| Value | Children | Probability | Fits node budget |", "|---:|---:|---:|---|"]
for candidate in prediction["candidates"]:
    candidate_table.append(f"| {candidate['value']} | {candidate['child_count']} | "
                           f"{candidate['probability']:.4f} | {candidate['fits_node_budget']} |")
display(Markdown("\n".join(candidate_table)))
fig = draw_records(demo["records"], prefix_length, "Held-out subtree: blue prefix → orange next node")
fig.savefig(OUT / "held_out_subtree.png", dpi=150)
plt.show()

continuation = generate(prefix, seed=cfg.seed + 200)
print("Generated continuation:", [(r.value, r.child_count) for r in continuation["records"][prefix_length:]])
fig = draw_records(continuation["records"], prefix_length, "Sampled continuation of the same prefix")
fig.savefig(OUT / "generated_subtree.png", dpi=150)
plt.show()
save_json(OUT / "prediction_example.json", dict(shape_id=demo["shape_id"],
    prefix=[asdict(r) for r in prefix], observed_next=asdict(actual), prediction=prediction,
    continuation=serializable_generation(continuation)))


## 10. Reload and verify inference

The checkpoint includes architecture settings, weights, the selected epoch,
source fingerprint, split IDs, and software/device information. To infer in a
fresh kernel, run the imports/configuration and the definition cells for node
records, the model, and inference; skip the extraction/training/demo execution
lines. Then call `load_checkpoint`, followed by `predict_next(..., net=loaded)`.
The notebook configuration's value/count vocabulary must match the checkpoint.

Artifacts contain the complete metrics, including any failed learning criterion.
Constrained validity is checked separately from learned predictive performance.
The full default profile is intended for the A100; a CPU smoke run verifies the
implementation without certifying GPU execution or the full experiment result.

In [ ]:
def load_checkpoint(path=OUT / "checkpoint.pt", destination="cpu"):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint.get("version") != 1:
        raise ValueError("Unsupported checkpoint version.")
    config = Config(**checkpoint["config"])
    loaded = NodeTransformer(config).to(destination)
    loaded.load_state_dict(checkpoint["state_dict"])
    loaded.eval()
    return loaded

@torch.no_grad()
def check_inference_and_reload():
    row = examples["test"][0]
    cut = min(4, len(row["records"]) - 1)
    batch = to_device(collate([row]))
    expected = model(batch)[0, cut].float().cpu()
    actual = next_logits(row["records"][:cut])
    torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-5)
    loaded = load_checkpoint()
    torch.testing.assert_close(next_logits(row["records"][:cut], loaded), actual, rtol=1e-4, atol=1e-4)
    prediction = predict_next(row["records"][:cut], net=loaded)
    assert all(0 <= item["probability"] <= 1 for item in prediction["candidates"])
    assert sum(item["probability"] for item in prediction["candidates"]) <= 1 + 1e-6
    expect_value_error(lambda: predict_next(row["records"], net=loaded))
    expect_value_error(lambda: generate([NodeRecord(0, 3)], max_nodes=2, net=loaded))
    complete = [NodeRecord(0, 0)]
    assert generate(complete, net=loaded)["records"] == complete
    for seed in (1, 2, 3):
        result = generate([], max_nodes=8, seed=seed, net=loaded)
        assert result["valid"] and len(result["records"]) <= 8
    checks["training_inference_alignment"] = True
    checks["checkpoint_reload_cpu"] = True
    print("PASS: inference alignment, CPU checkpoint reload, and generation boundaries.")

check_inference_and_reload()
save_json(OUT / "checks.json", checks)
print(f"Saved {PROFILE} experiment to {OUT.resolve()}")
print("Full learning criterion:",
      "not assessed by the smoke profile" if PROFILE == "smoke" else
      ("PASS" if learning_check["criterion_met"] else "NOT MET — see metrics and curves"))
